# picomake - mlp

In [2]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().split()
len(words)

32033

In [4]:
# build vocab stoi and itos dictionaries for character level indexing 

chars = set(sorted(letter for w in words for letter in w))

stoi = {s:i+1 for i,s in enumerate(sorted(chars))}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

### what am i doing? 

<small>

in the next cell, we shall build out the context window. its a custom set rolling window across each word. lets say context_size = 3, and the word is emma. the first window produces '000' next one produces, '005' and so on using stoi. X tensor is basically the whole set of this. Y tensor is the next character that is expected after the context window. 

after that, we use an embedding matrix of randomly initialised values to index the values to get the nn to work correctly. if a - 1 and z - 27, and the neural net learns a and z via the relative distance between the two. and in real language semantics, this contains no meaning. thats why we have an embedding table.

</small>

In [26]:
# build out dataset

context_size = 3
X = []
Y = []

firstword = True
for w in words:
    context = [0]*context_size

    for i in w+'.':
        ix = stoi[i]
        X.append(context)
        Y.append(ix)
        if firstword:
            print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context [1:] + [ix]
    firstword = False

X = torch.tensor(X) # X stores the context of one training example in chunk of context size (stoi)
Y = torch.tensor(Y) # Y stores the stoi value of each char in the word


... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .


In [ ]:
print(X.shape)
print(Y.shape)

# how the word 'emma' is stored in x:
X[:5]


torch.Size([228146, 3])
torch.Size([228146])


tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [21]:
# let us create an embedding matrix where every character 
# gets a 2 dimensional embedding using the awesome pytorch indexing

C = torch.randn((27,2))
emb = C[X]


print(f"c for emma  = {C[:5]}")
print(f"emb for emma = {emb[:5]}")

c for emma  = tensor([[ 1.1651, -0.5914],
        [-0.1032, -1.2893],
        [ 0.3337,  0.5309],
        [ 0.8997, -0.5117],
        [-0.0375, -0.2484]])
emb for emma = tensor([[[ 1.1651, -0.5914],
         [ 1.1651, -0.5914],
         [ 1.1651, -0.5914]],

        [[ 1.1651, -0.5914],
         [ 1.1651, -0.5914],
         [ 0.4197, -0.8437]],

        [[ 1.1651, -0.5914],
         [ 0.4197, -0.8437],
         [ 1.5876, -0.0741]],

        [[ 0.4197, -0.8437],
         [ 1.5876, -0.0741],
         [ 1.5876, -0.0741]],

        [[ 1.5876, -0.0741],
         [ 1.5876, -0.0741],
         [-0.1032, -1.2893]]])


### lets breakdown for the first training example

<small>

so first, we can see the **X tensor values for 'emma'** is :

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

**now C for 'emma'** is :

c for emma  = tensor([[ 1.1651, -0.5914],
        [-0.1032, -1.2893],
        [ 0.3337,  0.5309],
        [ 0.8997, -0.5117],
        [-0.0375, -0.2484]])

**now emb values for 'emma'** is :

emb for emma = tensor([[[ 1.1651, -0.5914],
         [ 1.1651, -0.5914],
         [ 1.1651, -0.5914]],

        [[ 1.1651, -0.5914],
         [ 1.1651, -0.5914],
         [ 0.4197, -0.8437]],

        [[ 1.1651, -0.5914],
         [ 0.4197, -0.8437],
         [ 1.5876, -0.0741]],

        [[ 0.4197, -0.8437],
         [ 1.5876, -0.0741],
         [ 1.5876, -0.0741]],

        [[ 1.5876, -0.0741],
         [ 1.5876, -0.0741],
         [-0.1032, -1.2893]]])

as you can see, x tensor values was [0,0,0] at 0 index, when C[X] done, it gave [C[0], C[0], C[0]] which is :

        [[ 1.1651, -0.5914],
         [ 1.1651, -0.5914],
         [ 1.1651, -0.5914]],

this is how the embeddings are used for representation learning. in the coming cells, I will probably backprop and update our embedding matrix as the model has to inherently learn the vector space representation for each letter by itself so predicting is free from any IDs associated with that letter :)

*quite the advantage :)*


</small>

In [22]:
print(emb.shape)

torch.Size([228146, 3, 2])


In [ ]:
# first hidden layer

# incoming dims = torch.Size([228146, 3, 2])

W1 = torch.randn((6, 100)) # 6,100 wants to get multiplied to torch.Size([5, 3, 2]) first word say
b1 = torch.randn(100)


# we can do torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1) or

# torch.cat(torch.unbind(emb, 1), 1).shape # unbind removes the dimension and flattens it and cat concatenates along the flattened dimension

# lets use .view for the tensor for ease

h1 = emb.view(-1, 6) @ W1 + b1 # -1 means any rows
h1 = torch.tanh(h1)
h1.shape


torch.Size([228146, 100])

In [24]:
# second hidden layer (last layer)

# incoming dims = torch.Size([228146, 100])

W2 = torch.randn(100, 27)
b2 = torch.randn(27)

h2 = h1@W2 + b2

# softmax

h2 = h2.exp() # logits
probs = h2/h2.sum(1, keepdim=True)
probs.shape

torch.Size([228146, 27])